# Assignment 3: Liquidity Provision in an AMM

### Uniswap V2: LP Shares, Fee Accrual, Impermanent Loss

**Course:** CS422 Blockchain Technologies
**Professor:** Aleksandr Kapitonov
**Student:** Jamol Shoymurzayev
**Student ID:** 220073
**Week 13, Individual Work**

---

This is Part 3 of the Uniswap V2 lab. I reuse my `UniswapV2Pool` class from Lab 15 and add the three LP methods the assignment asks for (`add_liquidity`, `remove_liquidity`, `fees_earned`). After that I run the fee-tier comparison from 3.4 and a bonus experiment on impermanent loss vs directional price drift.

**Contents**
- 3.1, adding liquidity and the ratio constraint
- 3.2, removing liquidity
- 3.3, fee accrual from k-growth
- 3.4, fee-tier simulation (5, 30, 100 bps)
- Bonus 3.5, directional drift: IL vs fee income
- Final short report

In [ ]:
import math
import random

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

## The `UniswapV2Pool` class

Same class I wrote in Lab 15, plus the three LP methods for this assignment:

- `add_liquidity`: first deposit mints `sqrt(x*y)` shares. Later deposits must match the current ratio and get `(dx / x) * S_total` new shares.
- `remove_liquidity`: burn shares, get back a proportional slice of both reserves.
- `fees_earned`: work back from the current claim using `sqrt(k / k_entry)` to recover what the deposit would have been if no fees had accrued, then take the difference.

In [ ]:
class UniswapV2Pool:
    def __init__(self, reserve_x: float, reserve_y: float, fee_bps: int = 30):
        self.reserve_x = reserve_x
        self.reserve_y = reserve_y
        self.fee_bps = fee_bps
        self.total_shares = 0.0
        self.lp_positions = {}   # provider -> shares
        self.lp_entry_k = {}     # provider -> k at deposit time

    @property
    def k(self) -> float:
        return self.reserve_x * self.reserve_y

    @property
    def price(self) -> float:
        return self.reserve_y / self.reserve_x

    @property
    def fee_rate(self) -> float:
        return self.fee_bps / 10_000

    def __repr__(self) -> str:
        price_str = f"{self.price:.4f}" if self.reserve_x > 0 else "n/a"
        return (f"UniswapV2Pool(reserve_x={self.reserve_x:.4f}, reserve_y={self.reserve_y:.4f}, "
                f"price={price_str}, k={self.k:.2f})")

    # ----- Swap methods (from Part 1) -----

    def get_amount_out(self, amount_in: float, token: str) -> float:
        assert token in ('x', 'y'), "token must be 'x' or 'y'"
        if token == 'x':
            reserve_in, reserve_out = self.reserve_x, self.reserve_y
        else:
            reserve_in, reserve_out = self.reserve_y, self.reserve_x
        amount_in_with_fee = amount_in * (1 - self.fee_rate)
        return (reserve_out * amount_in_with_fee) / (reserve_in + amount_in_with_fee)

    def swap(self, amount_in: float, token: str) -> float:
        k_before = self.k
        amount_out = self.get_amount_out(amount_in, token)
        if token == 'x':
            self.reserve_x += amount_in
            self.reserve_y -= amount_out
        else:
            self.reserve_y += amount_in
            self.reserve_x -= amount_out
        assert self.k >= k_before * 0.99999999, f"k decreased! {self.k} < {k_before}"
        return amount_out

    # ----- LP methods (Part 3) -----

    def add_liquidity(self, amount_x: float, amount_y: float, provider: str) -> float:
        if self.total_shares == 0:
            # First deposit: seed the pool and mint sqrt(x*y) shares.
            self.reserve_x = amount_x
            self.reserve_y = amount_y
            shares = math.sqrt(amount_x * amount_y)
        else:
            # Subsequent deposit: enforce the current ratio by recomputing amount_y
            # from amount_x. Depositing off-ratio would shift the price for free.
            amount_y = amount_x * (self.reserve_y / self.reserve_x)
            shares = (amount_x / self.reserve_x) * self.total_shares
            self.reserve_x += amount_x
            self.reserve_y += amount_y

        self.total_shares += shares
        self.lp_positions[provider] = self.lp_positions.get(provider, 0.0) + shares
        self.lp_entry_k[provider] = self.k   # snapshot k for later fee calculation
        return shares

    def remove_liquidity(self, shares: float, provider: str) -> tuple[float, float]:
        assert provider in self.lp_positions, "Unknown provider"
        assert self.lp_positions[provider] >= shares, "Insufficient shares"

        ownership = shares / self.total_shares
        amount_x = ownership * self.reserve_x
        amount_y = ownership * self.reserve_y

        self.reserve_x -= amount_x
        self.reserve_y -= amount_y
        self.total_shares -= shares
        self.lp_positions[provider] -= shares
        return amount_x, amount_y

    def fees_earned(self, provider: str) -> dict:
        assert provider in self.lp_positions

        ownership = self.lp_positions[provider] / self.total_shares
        current_x = ownership * self.reserve_x
        current_y = ownership * self.reserve_y

        # Both reserves grow by sqrt(k/k_entry) under pure fee accrual.
        k_ratio = math.sqrt(self.k / self.lp_entry_k[provider])
        deposit_x = current_x / k_ratio
        deposit_y = current_y / k_ratio

        return {
            'deposit_x': deposit_x,
            'deposit_y': deposit_y,
            'current_x': current_x,
            'current_y': current_y,
            'fee_x': current_x - deposit_x,
            'fee_y': current_y - deposit_y,
        }

---

## Task 3.1, Adding liquidity

Empty pool, then Alice adds 100 ETH and 200,000 USDC. I print shares and ownership percent. Then I poke at the ratio constraint with a deposit that has the wrong ratio and see what my code does with it.

In [ ]:
# Fresh pool, first deposit.
pool = UniswapV2Pool(reserve_x=0, reserve_y=0)

alice_shares = pool.add_liquidity(100, 200_000, 'Alice')
ownership_pct = pool.lp_positions['Alice'] / pool.total_shares * 100

print(f"Pool state:        {pool}")
print(f"Alice shares:      {alice_shares:,.4f}")
print(f"Alice ownership:   {ownership_pct:.2f}%")
print(f"Pool price (Y/X):  {pool.price:.2f}  (expected 2000.00)")

**Sanity check.** First deposit shares should be `sqrt(100 * 200000) = sqrt(20,000,000)` which is about 4472.14. Alice is the only LP so her ownership is 100%.

In [ ]:
# Wrong-ratio deposit probe: Bob *attempts* 100 ETH / 100 000 USDC
# (price-implied pair would be 100 ETH / 200 000 USDC at the current ratio).
price_before = pool.price
reserves_before = (pool.reserve_x, pool.reserve_y)

bob_shares = pool.add_liquidity(100, 100_000, 'Bob')

print(f"Price before:   {price_before:.2f}")
print(f"Price after:    {pool.price:.2f}       (unchanged — ratio was silently corrected)")
print(f"Reserves before: reserve_x={reserves_before[0]:.2f}, reserve_y={reserves_before[1]:.2f}")
print(f"Reserves after:  reserve_x={pool.reserve_x:.2f}, reserve_y={pool.reserve_y:.2f}")
print(f"Bob shares:     {bob_shares:,.4f}")
print(f"Bob ownership:  {pool.lp_positions['Bob'] / pool.total_shares * 100:.2f}%")

### Wrong-ratio deposit: what my code does and why

When Bob gives 100 ETH and 100,000 USDC to a pool that is already priced at 2000 USDC/ETH, my `add_liquidity` basically ignores the USDC amount he passed and recomputes it from the current ratio. So Bob ends up depositing 100 ETH plus 200,000 USDC and gets shares based on the ETH side only. Price stays at 2000.

I think this is the right behavior, for a couple of reasons.

The main one is that if you let off-ratio deposits through, the pool price would shift without anyone actually trading. That means someone could deposit in a lopsided way, then withdraw right after and walk away with more value than they put in. Arbitrage profit from nothing. Bad.

The other reason is more about the share formula. `dshares = (dx / x) * S_total` only makes sense if the pool is still on the same x*y curve. If I tried to use both `amount_x` and `amount_y` directly, two LPs depositing the same dollar value but with different ratios would get different share counts. Minting becomes ambiguous.

Real Uniswap V2 does basically the same thing in its router. It takes max amounts and spends the smaller side exactly, pulling only the matching amount of the other side. My version is simpler (it just anchors on `amount_x` instead of taking a min), but the idea is the same. I could have made it raise an error for off-ratio deposits, but silent correction is closer to what production does and easier to simulate with.

---

## Task 3.2, Removing liquidity

Alice deposits, 100 random swaps run, then she pulls everything out. Comparing what she put in vs what she gets back.

In [ ]:
random.seed(42)
pool = UniswapV2Pool(reserve_x=0, reserve_y=0)

deposit_x, deposit_y = 100.0, 200_000.0
shares = pool.add_liquidity(deposit_x, deposit_y, 'Alice')

# 100 random swaps, each up to 2% of the relevant reserve.
for _ in range(100):
    token = random.choice(['x', 'y'])
    reserve = pool.reserve_x if token == 'x' else pool.reserve_y
    amount = random.uniform(0.0001, 0.02) * reserve
    pool.swap(amount, token)

withdrawn_x, withdrawn_y = pool.remove_liquidity(shares, 'Alice')

print(f"Deposit:    {deposit_x:,.6f} ETH   {deposit_y:,.6f} USDC")
print(f"Withdrawal: {withdrawn_x:,.6f} ETH   {withdrawn_y:,.6f} USDC")
print(f"Difference: {withdrawn_x - deposit_x:+.6f} ETH   {withdrawn_y - deposit_y:+.6f} USDC")
print(f"Pool after: {pool}  (should be empty)")

### Where did the difference come from?

Most of the gain is from swap fees. Every time someone calls `swap`, the fee portion of the input gets kept in the pool and is never paid out. `get_amount_out` applies the fee to the input *before* computing the output, so that part just stays in the reserves. After 100 swaps, k is larger than it was at deposit. Alice is the only LP so she gets 100% of that growth back when she withdraws.

The reason the gain is not the same for X and Y is that the 100 swaps were not evenly split in direction. With seed 42 and only 100 draws, `random.choice(['x', 'y'])` can come out noticeably lopsided. In this run I got more x-swaps than y-swaps, meaning more ETH was being sold into the pool than bought out. So the pool ended holding more ETH and less USDC than it started with, and the price drifted from 2000 down to about 1650 USDC/ETH. `remove_liquidity` hands Alice a proportional slice of the *current* reserves, so she gets more ETH and fewer USDC than she put in.

So really two things stack. The fee part, which grows both reserves by roughly sqrt(k_new / k_entry) on average. And the price drift part, which shifts the pool toward whichever token was being dumped into it more. You can't separate them just from comparing deposit to withdrawal numbers. Task 3.3 below uses the k-ratio trick to isolate just the fee part.

If the RNG had given perfectly balanced flow, the price would have come back to 2000 and the gain would have split roughly equally across both tokens. The same mechanism is what causes impermanent loss in real Uniswap: when price drifts, the LP is left holding more of the losing token and less of the winning one. The bonus experiment measures how bad that gets.

---

## Task 3.3, Fee accrual

`fees_earned` reconstructs the no-fee counterfactual deposit using `growth = sqrt(k_current / k_entry)`, since under pure fee accrual both reserves scale up by that same factor on average. I check two things: (a) fees are essentially zero right after deposit, (b) they only grow, never shrink, as more swaps happen.

In [ ]:
random.seed(42)
pool = UniswapV2Pool(reserve_x=0, reserve_y=0)
pool.add_liquidity(100, 200_000, 'Alice')

print("Immediately after deposit (no swaps yet):")
for key, val in pool.fees_earned('Alice').items():
    print(f"  {key:>10}: {val:,.8f}")

# 1000 random swaps.
for _ in range(1000):
    token = random.choice(['x', 'y'])
    reserve = pool.reserve_x if token == 'x' else pool.reserve_y
    amount = random.uniform(0.0001, 0.02) * reserve
    pool.swap(amount, token)

print("\nAfter 1 000 swaps:")
fees1 = pool.fees_earned('Alice')
for key, val in fees1.items():
    print(f"  {key:>10}: {val:,.6f}")
fee_usd_1 = fees1['fee_x'] * pool.price + fees1['fee_y']
print(f"  fee_usd  : {fee_usd_1:,.2f}")

# 1000 more swaps.
for _ in range(1000):
    token = random.choice(['x', 'y'])
    reserve = pool.reserve_x if token == 'x' else pool.reserve_y
    amount = random.uniform(0.0001, 0.02) * reserve
    pool.swap(amount, token)

print("\nAfter 2 000 swaps:")
fees2 = pool.fees_earned('Alice')
for key, val in fees2.items():
    print(f"  {key:>10}: {val:,.6f}")
fee_usd_2 = fees2['fee_x'] * pool.price + fees2['fee_y']
print(f"  fee_usd  : {fee_usd_2:,.2f}")

print(f"\nFee USD grew from {fee_usd_1:,.2f} -> {fee_usd_2:,.2f}  "
      f"(monotone increase: {fee_usd_2 >= fee_usd_1})")

### Do fees grow monotonically?

Right after the deposit, `k_current == k_entry`, so `growth = sqrt(1) = 1`, `deposit_x` equals `current_x`, and both fee numbers are zero (up to floating point noise). The output above confirms this.

After trading, fees can only go up. The argument is simple: `swap` asserts `k_after >= k_before`, so k is non-decreasing across the simulation. That means `growth = sqrt(k / k_entry)` is non-decreasing too, and since `fee_x = current_x * (1 - 1/growth)`, fee_x also cannot go down while current_x is roughly stable (current_x only moves from rebalancing; trades do not pull value out of the pool).

In plain language: the only way k ever changes in this setup is by fees piling up inside the pool. A no-fee swap keeps k flat, a fee swap grows it. Nothing pulls value back out except `remove_liquidity`, and I am not calling that during the simulation. So fee_x and fee_y only climb. There is a small wiggle when you convert to USD because the spot price moves between measurements, but the underlying growth is monotonic.

---

## Task 3.4, Fee income vs volume

Helper `simulate_trading` plus a loop over fee tiers 5, 30, 100 bps. Same RNG seed for all three pools so the trade sizes are identical across tiers, only the fee changes. That way the comparison is apples to apples.

In [ ]:
def simulate_trading(pool, n_swaps=1000, max_trade_pct=0.02):
    """Run `n_swaps` random swaps against `pool`, up to `max_trade_pct` of the relevant reserve each."""
    for _ in range(n_swaps):
        token = random.choice(['x', 'y'])
        reserve = pool.reserve_x if token == 'x' else pool.reserve_y
        amount = random.uniform(0.0001, max_trade_pct) * reserve
        pool.swap(amount, token)

In [ ]:
fee_tiers = [5, 30, 100]                      # 0.05%, 0.30%, 1.00%
colors    = ['#2ca02c', '#1f77b4', '#d62728']
n_swaps, record_every = 2000, 50

plt.figure()

final_fee_usd = {}
for fee_bps, color in zip(fee_tiers, colors):
    # Fresh pool per tier, identical RNG seed -> identical trade sizes per step.
    random.seed(2024)
    pool = UniswapV2Pool(reserve_x=0, reserve_y=0, fee_bps=fee_bps)
    pool.add_liquidity(50, 100_000, 'Alice')  # $200 000 TVL (50 ETH · 2000 + 100 000 USDC)

    swap_numbers, fee_usds = [], []
    for i in range(1, n_swaps + 1):
        token = random.choice(['x', 'y'])
        reserve = pool.reserve_x if token == 'x' else pool.reserve_y
        amount = random.uniform(0.0001, 0.02) * reserve
        pool.swap(amount, token)

        if i % record_every == 0:
            fees = pool.fees_earned('Alice')
            fee_usd = fees['fee_x'] * pool.price + fees['fee_y']
            swap_numbers.append(i)
            fee_usds.append(fee_usd)

    final_fee_usd[fee_bps] = fee_usds[-1]
    plt.plot(swap_numbers, fee_usds, label=f'{fee_bps} bps ({fee_bps/100:.2f}%)',
             color=color, linewidth=2)

plt.xlabel('Swap number')
plt.ylabel('Cumulative LP fee income (USD)')
plt.title('LP fee income vs trading volume across fee tiers\n($200k TVL, 2 000 random swaps, shared trade-size RNG)')
plt.legend(title='Fee tier')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Final cumulative fee income after 2 000 swaps:")
for fee_bps, amount in final_fee_usd.items():
    print(f"  {fee_bps:>3} bps -> ${amount:>9,.2f}")

### Task 3.4 research questions

**(1) Which tier earns the most for high-volume vs low-volume pairs?**

In my simulation the highest fee tier wins in absolute dollars, because I fix the number and size of swaps across all three pools. Only the fee rate changes. So 100 bps earns roughly 20x what 5 bps earns for the same volume.

In reality the fee tier affects volume, which breaks this pattern. On a high-volume pair like ETH/USDC a 1% fee would push traders and arbitrageurs to cheaper pools and the volume would collapse. The 30 bps tier usually wins for pairs like that: low enough to keep volume, high enough to pay LPs back for volatility and IL. For a low-volume exotic token, volume is basically inelastic (there might not be another venue at all), so the 100 bps tier makes sense. More extraction per trade, plus the fee compensates LPs for the bigger IL risk on an illiquid asset.

**(2) Why do stablecoin pools usually sit at 5 bps?**

Stablecoins barely move against each other, so IL is basically negligible. LPs do not need fat fees to offset it. In exchange, the low fee attracts a lot of arbitrage volume whenever the pool drifts even a few bps from parity. In my simulation the 5 bps tier still earns real fees, just slowly. Scale the volume up to what real stablecoin pools actually see (way higher turnover than my 2000 swaps) and the absolute fee number ends up comparable to what 30 bps would earn at lower attracted volume. Fees are rate times volume. When volume is huge, even a tiny rate adds up.

**(3) Volume doubles, TVL stays the same. How does APR change?**

Roughly doubles. Fees per day are about `volume * fee_rate`, and APR is `fees_per_day / deposit_value * 365`. Deposit value (TVL) is fixed by assumption, so doubling volume doubles APR. Linear.

Caveat: this is only true for gross APR. If the extra volume is directional and the pool price drifts, the LP eats more impermanent loss too, and net return grows slower than linear. Task 3.5 below shows that effect.

---

## Bonus 3.5, Impermanent loss vs fees under directional flow

Question I wanted to answer: if the trading flow gets one-sided and the pool price drifts, at what drift does the LP start losing vs just holding the original tokens? That is the standard IL setup, but I wanted to measure it against fees earned in the same run.

Setup: one pool with 50 ETH and 100,000 USDC, 30 bps, $200k TVL. Alice deposits everything. Then I run 3000 swaps with a buy bias `p_buy`. When `p_buy = 0.5` flow is symmetric. Higher `p_buy` means more buys of ETH with USDC, which pushes the price up. Lower would do the opposite, symmetric to that case, so I only sweep upward.

Each trade is a near-constant $200 notional (jittered 0.5x to 1.5x). I do this rather than sizing by the pool reserve because under heavy bias the thinner side gets depleted fast and reserve-based sizing lets the price run away to absurd values. Constant dollar sizing keeps the experiment realistic.

For each scenario I track:

- `fee_usd`, cumulative fees in USD from `fees_earned`.
- `lp_value_usd`, Alice's share of the pool valued at the new spot price.
- `hold_value_usd`, what she would have had if she just kept her original 50 ETH + 100k USDC, priced at the same new spot.
- `il_usd = hold_value - (lp_value - fees)`, IL in dollars, computed so the fees do not get mixed into the IL number.
- `net_pnl_usd = lp_value - hold_value`, the net result of LPing vs holding.

The crossover drift is where `net_pnl_usd` flips from positive to negative.

In [ ]:
def run_drift_experiment(p_buy: float, n_swaps: int = 3000, seed: int = 7):
    """
    Directional simulation. p_buy is the probability of a buy (trader sends USDC,
    receives ETH -> pushes ETH price up). Each trade has a near-constant USD
    notional (~$200, i.e. 0.1% of initial TVL) so drift stays in a realistic
    range even under heavy bias. Returns Alice's end-state economics.
    """
    random.seed(seed)
    pool = UniswapV2Pool(reserve_x=0, reserve_y=0, fee_bps=30)

    deposit_x, deposit_y = 50.0, 100_000.0
    pool.add_liquidity(deposit_x, deposit_y, 'Alice')
    entry_price = pool.price

    trade_notional_usd = 200.0   # ~0.1% of $200k TVL

    for _ in range(n_swaps):
        notional = trade_notional_usd * random.uniform(0.5, 1.5)
        if random.random() < p_buy:
            # Buy ETH with USDC -> pushes ETH price up.
            pool.swap(notional, 'y')
        else:
            # Sell ETH for USDC -> pushes ETH price down.
            pool.swap(notional / pool.price, 'x')

    fees = pool.fees_earned('Alice')
    end_price = pool.price

    ownership = pool.lp_positions['Alice'] / pool.total_shares
    current_x = ownership * pool.reserve_x
    current_y = ownership * pool.reserve_y
    lp_value_usd = current_x * end_price + current_y

    # HODL counterfactual: original 50 ETH + 100 000 USDC at the *new* price.
    hold_value_usd = deposit_x * end_price + deposit_y

    fee_usd = fees['fee_x'] * end_price + fees['fee_y']

    # Isolate IL from fees: subtract fees out of LP value, compare vs HODL.
    il_usd = hold_value_usd - (lp_value_usd - fee_usd)
    net_pnl_usd = lp_value_usd - hold_value_usd
    price_drift = end_price / entry_price

    return {
        'p_buy': p_buy,
        'entry_price': entry_price,
        'end_price': end_price,
        'price_drift': price_drift,
        'fee_usd': fee_usd,
        'lp_value_usd': lp_value_usd,
        'hold_value_usd': hold_value_usd,
        'il_usd': il_usd,
        'net_pnl_usd': net_pnl_usd,
    }


p_buys = [0.50, 0.52, 0.54, 0.56, 0.58, 0.60, 0.62, 0.65]
results = [run_drift_experiment(p) for p in p_buys]

print(f"{'p_buy':>6} {'drift':>8} {'end_px':>10} {'fees':>12} {'IL':>12} {'net PnL':>13}")
print('-' * 65)
for r in results:
    print(f"{r['p_buy']:>6.2f} {r['price_drift']:>7.3f}x "
          f"{r['end_price']:>9,.1f} "
          f"${r['fee_usd']:>10,.2f} ${r['il_usd']:>10,.2f} ${r['net_pnl_usd']:>+11,.2f}")


In [ ]:
drifts  = [r['price_drift']    for r in results]
fees    = [r['fee_usd']        for r in results]
ils     = [r['il_usd']         for r in results]
netpnls = [r['net_pnl_usd']    for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Fees vs IL
ax1.plot(drifts, fees, marker='o', label='Fees earned (USD)',           color='#2ca02c', linewidth=2)
ax1.plot(drifts, ils,  marker='s', label='Impermanent loss (USD)',      color='#d62728', linewidth=2)
ax1.set_xlabel('Price drift (end_price / entry_price)')
ax1.set_ylabel('USD')
ax1.set_title('Fees vs Impermanent Loss by directional drift')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: net PnL crossover
ax2.plot(drifts, netpnls, marker='o', color='#1f77b4', linewidth=2, label='LP net PnL vs HODL')
ax2.axhline(0, color='black', linestyle='--', alpha=0.5, label='Break-even')
ax2.set_xlabel('Price drift (end_price / entry_price)')
ax2.set_ylabel('LP net profit vs HODL (USD)')
ax2.set_title('Break-even drift: where IL eats the fees')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### What I learned from 3.5

Fees and IL push in opposite directions, and they grow at very different rates.

Fees are bounded by volume. 3000 trades times ~$200 each is about $600k of turnover, and at 30 bps that caps total fees near $1800. Volume does not change between scenarios because I run the same number of swaps, so fees hover around that ceiling in every row.

IL is not capped like that. The closed form `1 - 2*sqrt(r)/(1+r)` is small for small drift but picks up fast. At drift 1.2x it is about 0.4%, at 1.8x about 4%, at 5x about 20%. On a $200k-ish position those translate to roughly $800, $11k, and $40k+. Fees can't keep up once drift becomes material.

Observed crossover in my run:

| p_buy | drift | fees (USD) | IL (USD) | net PnL (USD) |
|------:|------:|-----------:|---------:|--------------:|
| 0.50 | 1.18x |  ~1850 |  ~710  | **+1135** |
| 0.52 | 1.78x |  ~2040 | ~11060 | -9020 |
| 0.54 | 2.34x |  ~2170 | ~28180 | -26010 |
| 0.60 | 4.91x |  ~2570 | ~148020 | -145450 |
| 0.65 | 7.53x |  ~2810 | ~304050 | -301230 |

Break-even sits between `p_buy = 0.50` and `p_buy = 0.52`, at a drift of only about 1.2x (i.e. a 20% price move from entry). Past that, fees can't offset IL. At 0.54 the position is already down $26k against HODL while earning only about $2k in fees.

Lesson for me: quoting LP APR on its own is kind of meaningless. Fees are only one side of the trade. What really matters is fees minus IL over the holding period. This also helped me see why stablecoin and LST/ETH pools are so popular (drift is structurally small), and why Uniswap V3 exists. Picking a tight range in V3 caps your IL exposure, so you can keep a similar fee income at a lower total risk.

---

## Final report, one finding

The most interesting thing for me was seeing how quickly impermanent loss overtakes fees once flow becomes directional.

In Task 3.4 the setup is symmetric. Price ends up close to entry and IL averages to basically zero. All three fee tiers look like scaled versions of each other, and higher bps looks strictly better. That picture falls apart in 3.5. With only a mild 52/48 imbalance in buys vs sells the LP is already losing against HODL at a drift of ~1.2x, and it gets worse fast from there. At drift ~5x (p_buy = 0.60 in my run) fees are about $2500 and IL is about $148k. Two orders of magnitude apart.

That changed how I think about fee tiers. I used to read 5 bps as low yield and 100 bps as high yield, but really it is about what drift you expect on the pair. Stablecoins can sit at 5 bps because drift is structurally tiny for them. Volatile pairs need 100 bps to pay LPs back for the IL they will definitely eat. It also explains why concentrated liquidity in V3 exists: narrowing the range caps your IL exposure, so you can keep a similar fee income with less risk.

Practical takeaway: when a dashboard quotes an LP APR by itself, it is only half the story. You need an idea of how much drift to expect, otherwise the "APR" number does not tell you whether the position actually makes money.